In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import os
os.environ['GLOG_minloglevel'] = '3'  # Suppress all logs below ERROR level

from dataclasses import field

from attr import dataclass
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from pathlib import Path

ROOT_DIR = Path("../../")

pose_options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path="../../models/mediapipe/pose_landmarker_heavy.task"
    ),
    running_mode=vision.RunningMode.VIDEO,
)

hand_options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path="../../models/mediapipe/hand_landmarker.task"
    ),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2
)

In [11]:
import cv2
import mediapipe as mp
from sortedcontainers import SortedDict
from src.utils.video import read_cap_segments
from typing import Any
import numpy as np
from src.shared.lm_processing.landmarks import Landmarks, nn_parser, make_hip_centric
from src.shared.utils.mediapipe import mp_to_arr
import json

MAX_CLIP_FRAME_SEPARATION = 1.0
BOUNDING_BOX_PADDING = 0.2
FPS = 6
MIN_CLIP_DURATION = 3.5
STATIC_THRESHOLD_PX_S = 13.5

class PersonResults:
    def __init__(self, id):
        self.id = id
        self.clips = []

    @dataclass
    class Clip:
        start: float
        end: float
        boxes: SortedDict[float, Any] = field(default_factory=SortedDict)
        max_box_size: dict[str, int] = field(default_factory=lambda: {"x": 0, "y": 0})

    def add_bounding_box_frame(self, timestamp, bounding_box):
        x_size = bounding_box[2] - bounding_box[0]
        y_size = bounding_box[3] - bounding_box[1]
        if len(self.clips) > 0 and self.clips[-1].end + MAX_CLIP_FRAME_SEPARATION > timestamp:
            self.clips[-1].boxes[timestamp] = bounding_box
            self.clips[-1].end = timestamp
            self.clips[-1].max_box_size["x"] = max(self.clips[-1].max_box_size["x"], x_size)
            self.clips[-1].max_box_size["y"] = max(self.clips[-1].max_box_size["y"], y_size)
        else:
            to_add = PersonResults.Clip(start=timestamp, end=timestamp, boxes=SortedDict({timestamp: bounding_box}), max_box_size={"x": x_size, "y": y_size})
            self.clips.append(to_add)

def process_folder(PATH):
    os.makedirs(ROOT_DIR / "data" / "processed" / "landmarks" / PATH.name / "unlabeled", exist_ok=True)
    def process_file(f, unlabeled):
        cap = cv2.VideoCapture(PATH / ("unlabeled" if unlabeled else "video") / f)

        with open(ROOT_DIR / "data" / "processed" / "bounding_boxes" / PATH.name / f.replace(".mp4", ".json"), "r") as bb_file:
            bounding_boxes = json.load(bb_file)

        # Start by creating the clips for each person
        people = {}
        for entry in bounding_boxes:
            for person in entry["boxes"].keys():
                if person not in people:
                    people[person] = PersonResults(person)
                people[person].add_bounding_box_frame(entry["timestamp"], entry["boxes"][person])

        output = {}
        # Now process each person's clip
        for person in people.values():
            output[person.id] = {}
            for clip_index, clip in enumerate(person.clips):
                lm = Landmarks(max_frames_interpolation=12)
                with vision.PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
                    vision.HandLandmarker.create_from_options(hand_options) as hand_landmarker: # Reset the model for each clip

                    static_status = 0 # 0 not checked, 1 is moving, 2 is static
                    last_position = None
                    checked_frames = 0
                    max_accel = 0

                    for frame, timestamp in read_cap_segments(cap, fps=FPS, start=clip.start, end=clip.end):
                        # Start by getting the current frame's bounding box
                        bounding_box_idx = clip.boxes.bisect_left(timestamp)
                        bounding_box_idx = max(0, bounding_box_idx - 1) # If its too early, use the first bounding box
                        bounding_box = clip.boxes.peekitem(bounding_box_idx)[1]
                        # Get the bounding box's center
                        x_center = bounding_box[0] + (bounding_box[2] - bounding_box[0]) / 2
                        y_center = bounding_box[1] + (bounding_box[3] - bounding_box[1]) / 2
                        # Calculate the update distance with paddings
                        x_distance_center = (clip.max_box_size["x"] * (1 + BOUNDING_BOX_PADDING)) / 2
                        y_distance_center = (clip.max_box_size["y"] * (1 + BOUNDING_BOX_PADDING)) / 2
                        # Calculate the corners
                        x_start = max(0, x_center - x_distance_center)
                        x_end = x_center + x_distance_center
                        y_start = max(0, y_center - y_distance_center)
                        y_end = y_center + y_distance_center
                        # Crop the image and make it contiguous in memory
                        frame = frame[int(y_start):int(y_end), int(x_start):int(x_end)]
                        frame = np.ascontiguousarray(frame)
                        # Process the image
                        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
                        pose_result = pose_landmarker.detect_for_video(mp_image, int(timestamp * 1000))
                        hand_result = hand_landmarker.detect_for_video(mp_image, int(timestamp * 1000))

                        # Parse it
                        pose_landmarks = pose_result.pose_landmarks[0] if len(pose_result.pose_landmarks) > 0 else None

                        left_hand_landmarks = None
                        right_hand_landmarks = None
                        for idx in range(len(hand_result.hand_landmarks)):
                            if idx < len(hand_result.handedness):
                                hand_category = hand_result.handedness[idx][0].category_name
                                if hand_category == "Right":
                                    right_hand_landmarks = hand_result.hand_landmarks[idx]
                                else:
                                    left_hand_landmarks = hand_result.hand_landmarks[idx]

                        # Add it to Landmarks
                        lm.add(pose_landmarks, left_hand_landmarks, right_hand_landmarks)

                        # Check if this is just a static image and we should stop processing
                        if static_status == 0 and pose_landmarks is not None:
                            pose_arr = mp_to_arr(pose_landmarks)  # (N,2) or (N,3)

                            # require hips to exist
                            if pose_arr.shape[0] <= 24:
                                # not enough landmarks — treat as unknown, continue collecting frames
                                last_position = None
                                last_timestamp = timestamp
                                checked_frames += 1
                                continue

                            # pelvis in normalized coords (0..1) relative to the cropped frame
                            left_hip = pose_arr[23][:2]
                            right_hip = pose_arr[24][:2]
                            pelvis_norm = (left_hip + right_hip) / 2.0

                            # convert to pixels (cropped frame)
                            h_crop, w_crop = frame.shape[:2]
                            pelvis_px = pelvis_norm * np.array([w_crop, h_crop])  # (x_px, y_px)

                            if last_position is not None and last_timestamp is not None:
                                dt = timestamp - last_timestamp
                                # guard against zero / tiny dt (skip sample)
                                if dt <= 1e-3:
                                    last_position = pelvis_px.copy()
                                    last_timestamp = timestamp
                                    continue

                                # instantaneous speed in pixels/second
                                vel = np.linalg.norm(pelvis_px - last_position) / dt

                                # update max (we keep the max velocity seen while deciding static/moving)
                                max_accel = max(max_accel, vel)
                                checked_frames += 1

                                # simple decision after a few frames (fast and deterministic)
                                if checked_frames >= FPS * MIN_CLIP_DURATION:
                                    if max_accel < STATIC_THRESHOLD_PX_S:
                                        static_status = 2  # static
                                        break
                                    else:
                                        static_status = 1  # moving

                            else:
                                # initialize
                                checked_frames += 1

                            last_position = pelvis_px.copy()
                            last_timestamp = timestamp

                        # Show it
                        cv2.imshow("test", frame)
                        cv2.waitKey(1)

                    if static_status == 2 or checked_frames < FPS * MIN_CLIP_DURATION: # If it lasts less than a second or is static, discard it
                        print("Discarded")
                        continue

                    # Now go over each landmark frame, pass it through nn_parser
                    parsed_landmarks = []
                    for pose, left, right, frame_num in lm.get_landmarks(continuous=False, return_frame_number=True):
                        parsed = nn_parser(pose, left, right)
                        parsed_landmarks.append((clip.start + frame_num / FPS, parsed))

                    # And save it
                    output[person.id][clip_index] = parsed_landmarks

        with open(ROOT_DIR / "data" / "processed" / ("landmarks/unlabeled" if unlabeled else "landmarks") / PATH.name / f.replace(".mp4", ".json"), "w") as out_file:
            json.dump(output, out_file)

    for f in os.listdir(PATH / "video"):
        print(f)
        process_file(f, False)
        break

process_folder(ROOT_DIR / "data" / "raw" / "0-AsociacionCivil")
cv2.destroyAllWindows()

0-100.mp4


I0000 00:00:1765748793.185873 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765748793.187962 1710709 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765748793.223940 1710713 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765748793.272481 1710728 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1765748793.273095 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765748793.275964 1710737 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765

Discarded


I0000 00:00:1765748806.202944 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765748806.205028 1711288 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765748806.240266 1711294 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765748806.286898 1711309 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1765748806.289762 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765748806.291435 1711316 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765

Discarded


...

Discarded
Discarded


I0000 00:00:1765749119.118574 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765749119.120751 1722426 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765749119.155324 1722431 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765749119.199830 1722445 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1765749119.204168 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765749119.205634 1722454 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765

Discarded


I0000 00:00:1765749119.423161 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765749119.426592 1722544 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765749119.463142 1722551 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765749119.506686 1722563 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1765749119.511962 1691886 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765749119.513552 1722573 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.3.1-arch1.2), renderer: AMD Radeon RX 570 Series (radeonsi, polaris10, ACO, DRM 3.61, 6.12.61-1-MANJARO)
W0000 00:00:1765

Discarded
